# Co-visiting de Lille — modèle simple Top-5

**Objectif :** recommander 5 lieux à partir d'un lieu de départ en utilisant **uniquement le nombre de co-visites historiques**. Aucune distance géographique, aucun coefficient temporel, aucun GNN.

**Important :** les données décrivent des relations de co-visiting **agrégées**, pas les déplacements individuels et chronologiques des touristes.

## 1. Charger le fichier et définir les paramètres

- `gid_from` : lieu de départ A
- `gid_to` : lieu associé B
- `NbPerMaxDurationDays_14` : nombre de co-visites enregistrées dans une fenêtre de **14 jours**, utilisé sans ajout de pondération temporelle
- `K = 5` : nombre de lieux recommandés

Place le fichier CSV dans le **même dossier** que ce notebook. Si besoin, change uniquement `CHEMIN_CSV`.

In [ ]:
from pathlib import Path
import pandas as pd

K = 5
POIDS = "NbPerMaxDurationDays_14"
CHEMIN_CSV = Path("MEL_circulationGraph_0(1).csv")
DOSSIER_RESULTATS = Path("resultats_co_visiting_seul")
DOSSIER_RESULTATS.mkdir(exist_ok=True)

# Cherche aussi le CSV dans le dossier parent si le notebook est placé ailleurs.
if not CHEMIN_CSV.exists():
    autre_chemin = Path("../MEL_circulationGraph_0(1).csv")
    if autre_chemin.exists():
        CHEMIN_CSV = autre_chemin

donnees = pd.read_csv(CHEMIN_CSV, sep=";")
donnees = donnees[donnees[POIDS] > 0].copy()
print("Nombre de lignes après filtrage :", len(donnees))
donnees[["gid_from", "gid_to", "year", POIDS]].head()

## 2. Découper les données selon les années

- **Jusqu'à 2016 :** construire les premières recommandations
- **2017 :** évaluer la méthode sur une année de validation
- **Jusqu'à 2017 :** reconstruire les recommandations pour le test final
- **2018–2019 :** vérifier les prédictions sur des données futures

`year <= 2016` inclut aussi les années antérieures à 2013 présentes dans le fichier.

In [ ]:
historique_2016 = donnees[donnees["year"] <= 2016]
validation_2017 = donnees[donnees["year"] == 2017]
historique_2017 = donnees[donnees["year"] <= 2017]
test_2018_2019 = donnees[donnees["year"].isin([2018, 2019])]

print("Historique <= 2016 :", len(historique_2016))
print("Validation 2017   :", len(validation_2017))
print("Historique <= 2017:", len(historique_2017))
print("Test 2018-2019   :", len(test_2018_2019))

## 3. Calculer le score de co-visiting

Pour chaque paire de lieux orientée **A → B**, on additionne les nombres de co-visites de toutes les lignes historiques (années et pays).

\[
S(A,B)=\sum_{i\in\text{historique des relations } A\to B} w_i
\]

Exemple : si A → B a été compté 10, 15 puis 20 fois, le score vaut **45**.

On ne garde pas les relations **A → A**. En cas d'égalité de score, le code classe les destinations par leur identifiant `gid_to`, comme dans le script initial.

In [ ]:
def calculer_top5(historique):
    relations = historique[["gid_from", "gid_to", POIDS]].copy()
    relations = relations[relations["gid_from"] != relations["gid_to"]]

    # 1. Total des co-visites de chaque relation orientée A -> B.
    sommes = relations.groupby(["gid_from", "gid_to"], as_index=False)[POIDS].sum()

    # 2. Destinations avec le poids le plus élevé, pour chaque source.
    sommes = sommes.sort_values(
        ["gid_from", POIDS, "gid_to"], ascending=[True, False, True]
    )

    # 3. Les 5 destinations les mieux classées par source.
    top5_table = sommes.groupby("gid_from").head(K)
    top5_dictionnaire = top5_table.groupby("gid_from")["gid_to"].agg(list).to_dict()
    return top5_dictionnaire, top5_table

predictions_2017, classement_2017 = calculer_top5(historique_2017)
print("Nombre de lieux de départ avec un Top-5 :", len(predictions_2017))
classement_2017.head(10)

## 4. Gérer les lieux de départ inconnus

Quand un lieu de départ n'apparaissait pas dans l'historique, l'ancien code recommandait les **lieux globalement les plus populaires**. On conserve exactement cette règle pour obtenir le même score.

On prend les **6** destinations les plus fréquentes au total pour pouvoir en retirer une si elle correspond au lieu de départ, et garder 5 propositions.

In [ ]:
def calculer_populaires(historique):
    return historique.groupby("gid_to")[POIDS].sum().nlargest(6).index.tolist()

populaires_2017 = calculer_populaires(historique_2017)
print("Identifiants des lieux les plus populaires :", populaires_2017)

# Exemple : afficher le Top-5 réellement calculé pour un lieu présent.
lieu_exemple = next(iter(predictions_2017))
print("Lieu de départ :", lieu_exemple)
print("Ses 5 recommandations :", predictions_2017[lieu_exemple])

## 5. Tester les prédictions : la métrique Top-5

Pour chaque relation A → B observée pendant le test :

- `hit5 = 1` si B appartient aux 5 recommandations calculées à partir du passé ;
- `hit5 = 0` sinon.

Le score **pondéré** est :

\[
\mathrm{Top5}=\frac{\sum_i w_i\times\mathrm{hit5}_i}{\sum_i w_i}
\]

avec \(w_i\) = nombre de co-visites à 14 jours dans la ligne de test.

Une ligne représentant 100 co-visites a donc 100 fois plus de poids qu'une ligne représentant une seule co-visite.

In [ ]:
def evaluer(test, predictions, populaires):
    # Même exclusion des auto-relations que dans le code original.
    details = test[test["gid_from"] != test["gid_to"]].copy()
    succes = []

    for depart, arrivee in zip(details["gid_from"], details["gid_to"]):
        top5 = predictions.get(depart) or [x for x in populaires if x != depart][:K]
        succes.append(int(arrivee in top5))

    details["hit5"] = succes

    score = (details["hit5"] * details[POIDS]).sum() / details[POIDS].sum()
    source_connue = details["gid_from"].isin(predictions)
    couverture = details.loc[source_connue, POIDS].sum() / details[POIDS].sum()
    return score, couverture, details

# Validation 2017 : recommandations faites avec le passé jusqu'à 2016.
predictions_2016, _ = calculer_top5(historique_2016)
populaires_2016 = calculer_populaires(historique_2016)
score_validation, _, _ = evaluer(validation_2017, predictions_2016, populaires_2016)
print(f"Top-5 validation 2017 : {score_validation:.3%}")

## 6. Test final sur 2018–2019

**Aucune observation de 2018 ou 2019 n'entre dans la création des recommandations.** Ces années servent uniquement à mesurer si les recommandations retrouvent les relations observées.

In [ ]:
score_test, couverture, details_test = evaluer(
    test_2018_2019, predictions_2017, populaires_2017
)

print(f"Top-5 test 2018-2019 : {score_test:.3%}")
print("Nombre de relations (lignes) testées :", len(details_test))
print("Somme des poids testés :", int(details_test[POIDS].sum()))
print(f"Couverture pondérée des lieux de départ : {couverture:.2%}")

details_test[["gid_from", "gid_to", "year", "country", POIDS, "hit5"]].head(10)

## 7. Exporter les résultats

Les résultats sont enregistrés dans le dossier `resultats_co_visiting_seul` :

- `resultats_top5.csv` : score et indicateurs globaux.
- `test_details_base.csv` : chaque relation testée et son `hit5`.

Ces colonnes et calculs reproduisent la sortie du script simplifié initial.

In [ ]:
details_test[["gid_from", "gid_to", "year", "country", POIDS, "hit5"]].to_csv(
    DOSSIER_RESULTATS / "test_details_base.csv", index=False
)

resultats = pd.DataFrame([{
    "modele": "base",
    "top5_test": score_test,
    "top5_test_pct": round(score_test * 100, 3),
    "nb_lignes_test": len(details_test),
    "poids_test": int(details_test[POIDS].sum()),
    "source_vues_train_pct": round(couverture * 100, 2),
}])
resultats.to_csv(DOSSIER_RESULTATS / "resultats_top5.csv", index=False)
resultats

## À savoir expliquer à l'oral

**Quel algorithme ?** Une recommandation Top-5 par **somme des fréquences de co-visiting** : c'est une baseline heuristique, pas un modèle neuronal.

**Comment les cinq lieux sont-ils choisis ?** On classe les destinations par total de co-visites historiques depuis un lieu donné.

**Que mesure le 16,1 % ?** La part **pondérée par les co-visites** des relations de test dont la destination se trouve dans le Top-5 historique.

**Limite :** la prédiction porte sur une *relation agrégée* et non sur le prochain lieu réellement visité par un touriste.